<a href="https://colab.research.google.com/github/HafizRizqi/MachineLearning_2026/blob/main/JS09/JS09_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🏐Lab 4
ANNOY vs. FAISS vs. HNSW

perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.

In [2]:
	# Install ANNOY
	!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 9.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=034fe3992221a8c31b6f422e06a20b32a0ad5c2511e72f6f35a115cb369ee38d
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [4]:
	!pip install faiss-cpu
	!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 60.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 16.7 MB/s  0:00:17
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 101.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 68.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 65.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [faiss-gpu]


In [6]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=27c3d326d71fd9649a4595eadceac088d3b60d47d8228b58db40ded5c68697c9
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [7]:
	import numpy as np
	import time
	from annoy import AnnoyIndex
	import faiss
	import hnswlib

	# ===============================
	# Build 1mio rows with 5D
	# ===============================
	n_data = 1_000_000   # try 100_000 if you have limited memory
	dim = 5
	X = np.random.random((n_data, dim)).astype(np.float32)

	# Query point
	query = np.random.random((1, dim)).astype(np.float32)
	k = 10

	# ===============================
	# 2. Annoy
	# ===============================
	print("=== Annoy ===")
	ann_index = AnnoyIndex(dim, 'euclidean')

	start = time.time()
	for i in range(n_data):
	    ann_index.add_item(i, X[i])
	ann_index.build(10)  # 10 trees
	build_time = time.time() - start

	start = time.time()
	neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", neighbors[0][:5], "...")

	# ===============================
	# 3. FAISS (Flat Index)
	# ===============================
	print("\n=== FAISS (IndexFlatL2) ===")
	faiss_index = faiss.IndexFlatL2(dim)

	start = time.time()
	faiss_index.add(X)
	build_time = time.time() - start

	start = time.time()
	distances, indices = faiss_index.search(query, k)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", indices[0][:5], "...")

	# ===============================
	# 4. HNSW (hnswlib)
	# ===============================
	print("\n=== HNSW (hnswlib) ===")
	hnsw_index = hnswlib.Index(space='l2', dim=dim)

	start = time.time()
	hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
	hnsw_index.add_items(X)
	build_time = time.time() - start

	hnsw_index.set_ef(50)

	start = time.time()
	labels, distances = hnsw_index.knn_query(query, k=k)
	query_time = time.time() - start

	print("Build time:", build_time, "detik")
	print("Query time:", query_time, "detik")
	print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 22.885491847991943 detik
Query time: 0.0003108978271484375 detik
Neighbors: [167638, 236528, 962976, 541197, 530929] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.015258312225341797 detik
Query time: 0.011728763580322266 detik
Neighbors: [167638 236528 311815 962976 530929] ...

=== HNSW (hnswlib) ===
Build time: 174.4939444065094 detik
Query time: 0.00022459030151367188 detik
Neighbors: [167638 236528 311815 962976 541197] ...
